In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONHASHSEED'] = '2'

In [ ]:
import numpy as np
np.random.seed(18)

import matplotlib.pyplot as plt
import time

import tensorflow as tf
from tensorflow.python.client import device_lib
from tensorflow.keras.optimizers import Adam
tf.random.set_seed(18)
tf.keras.utils.set_random_seed(18)

from func_file_Data import *
from func_file_Model import *

## Training with incremental learning

### Data generation

#### Set the data generation parameters

In [ ]:
#Whether to visualize distributions and example images
run_visualization_check = True

In [ ]:
#Matrix field size:
fixed_size = 50
upscale_factor = 8

if run_visualization_check:
    print("Input shape:   " + str(fixed_size) + "x" + str(fixed_size))
    print("Output shape: " + str(fixed_size*upscale_factor) + "x" + str(fixed_size*upscale_factor))

In [ ]:
#Number of emitters distribution: (a Lin-in-Lin triangle distribution)
LB_concentration = 1            #Lower bound on the triangle distribution of emitter numbers
UB_concentration = 100          #Upper bound on the triangle distribution of emitter numbers

if run_visualization_check:
    plt.figure(figsize=(6,4))
    plt.hist(Emit_num_dist(LB_concentration, UB_concentration, 10000), UB_concentration-LB_concentration+1)
    plt.title("Concentration histogram")
    plt.show()

In [ ]:
#Emittor intensity & Camera Noise intensity: (a Lin-in-Log triangle distribution)
EI_lower_bound = 1               #Lower_bound in emit_intensity_dist
EI_upper_bound = 4               #Upper_bound in emit_intensity_dist

CNI_lower_bound = 0              #Lower_bound in noise_intensity_dist for camera shot noise
CNI_upper_bound = 2              #Upper_bound in noise_intensity_dist for camera shot noise

if run_visualization_check:
    plt.figure(figsize=(12,4))
    plt.subplot(121)
    plt.hist(Emit_intensity_dist(EI_lower_bound, EI_upper_bound, 10000), 25)
    plt.title("Average emitter intensity [10**] histogram")
    plt.subplot(122)
    plt.hist(Noise_intensity_dist(CNI_lower_bound, CNI_upper_bound, 10000), 25)
    plt.title("Average camera noise [10**] histogram")
    plt.show()

In [ ]:
#Point spread function width: (a Lin-in-Log triangle distribution)
PSF_lower_bound = 0.1            #Lower_bound in PSF_dist; related to amplitude, not intensity
PSF_upper_bound = 1              #Upper_bound in PSF_dist; related to amplitude, not intensity

if run_visualization_check:
    plt.figure(figsize=(6,4))
    plt.hist(PSF_dist(PSF_lower_bound, PSF_upper_bound, 10000), 25)
    plt.title("PSF width [10**] histogram")
    plt.show()

#### Generate a data batch

In [ ]:
data_generation_parameters = [fixed_size, upscale_factor, LB_concentration, UB_concentration, EI_lower_bound, EI_upper_bound, CNI_lower_bound, CNI_upper_bound, PSF_lower_bound, PSF_upper_bound]
data_w_blur_check, data_clear_check = Get_data(10, data_generation_parameters)

if run_visualization_check:
    print("Input data shape: ", data_w_blur_check.shape)
    print("Output data shape:", data_clear_check.shape)
    
    plt.figure(figsize=(15,5))
    plt.subplot(131)
    plt.matshow(data_w_blur_check[0], cmap="gray", fignum=False)
    plt.subplot(132)
    plt.matshow(data_clear_check[0].reshape(fixed_size,upscale_factor,fixed_size,upscale_factor).sum(axis=(1,3)), cmap="gray", fignum=False)
    plt.subplot(133)
    plt.matshow(data_clear_check[0], cmap="gray", fignum=False)

## Train the model from scratch

### Build the model

Tensorflow may print warnings despite working correctly.

In [ ]:
#Build the ResNet with 8x upsampling
channels = 64
blocks = [3,3,3,3]

model = ResNet_model_paper(channels, blocks)

In [ ]:
#Set the default 50x50 shape
_ = model(tf.random.normal([1, 50, 50, 1]))

In [ ]:
#Model summary
model.summary()

### Set the training parameters

In [ ]:
#Parameters
batch_size = 8           #Batch size, number of samples processed in broadcasting
GA_steps = 16            #Gradient Acummulation steps, number of batch sizes processed for one gradient descend step
#Note: choose the batch_size according to your GPU memory and correspondingly adjust the GA_steps

num_of_epochs = 10000    #Arbitrarily large number; use the logger and train till convergence (around 500 in our case)
data_per_epoch = 5120    #16 batch * 8 GA * 40 gradient steps; not to break the GA accumulation with incomplete sets
data_for_val = 1280      #16 batch * 8 GA * 10 steps

In [ ]:
#Initial learning rate value
initial_lr = 1e-4

#Reduce learning rate on plateau callback
reduce_lr_factor = 0.5
reduce_lr_patience = 50

In [ ]:
#Loss function
reg_strength = 1e-1                                #Entropy regularization strength
used_loss = Custom_CCE_conv_func_regularized       #Either CCE or MSE; CCE can be used effectivelly even for regression task since the output behaves as probability distribution
loss_name = "custom_cce_conv_func_regularized"     #Name for correct logging

#Metric function
used_metric = Custom_mae_conv_func
metric_name = "custom_mae_conv_func"               #Name for correct logging
metric_name_val = "val_custom_mae_conv_func"       #Name for correct logging

### Training the model using custom manual loop

Tensorflow may print (an ungodly amount of) warnings despite working correctly. <br>
The training can take a few days, depending on your computational resources, amount of training data, chosen device-agnostic parameter ranges, and the network complexity.

In [ ]:
#Number of processors to simultaneously generate data samples
datagen_workers = 8

#Directory for logger to monitor the progress
starting_datetime = time.strftime("%Y_%m_%d-%H_%M_%S")
if not os.path.exists("Log_files"):
    os.mkdir("Log_files")
os.mkdir("Log_files/" + starting_datetime)

# Loss, metric, and optimizer
loss_fn = used_loss
metric_fn = used_metric
optimizer = Adam(learning_rate = initial_lr, 
                 use_ema = True)

# Initialize logger
log_file = "Log_files/" + starting_datetime + "/Training_log.txt"
logger = EpochLoggerCallback(log_file=log_file, monitor=metric_name_val, optimizer=optimizer, mode="min")
logger.set_model(model)

#Start
best_val_loss = np.inf
wait = 0

for epoch in range(num_of_epochs):
    print(f"\nEpoch {epoch+1}/{num_of_epochs}")
    
    # Generate training + validation data
    X_train, y_train = Generate_epoch_data(data_per_epoch, data_generation_parameters, workers=datagen_workers)
    X_val, y_val     = Generate_epoch_data(data_for_val, data_generation_parameters, workers=datagen_workers)
    
    # Initialize accumulators
    accum_grads = [tf.zeros_like(var, dtype=tf.float32) for var in model.trainable_variables]
    step = 0
    
    train_metric_total = 0.0
    train_samples = 0
    
    val_metric_total = 0.0
    val_samples = 0
    
    #Training in current epoch
    for i in range(0, len(X_train), batch_size):
        x_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        with tf.GradientTape() as tape:
            y_pred = model(x_batch, training=True)
            loss_per_sample = loss_fn(y_batch, y_pred, reg_strength_=reg_strength)
            loss = tf.reduce_mean(loss_per_sample)

        grads = tape.gradient(loss, model.trainable_variables)

        # Accumulate gradients
        accum_grads = [ag + g for ag, g in zip(accum_grads, grads)]

        step += 1
        if step % GA_steps == 0:
            # Average accumulated gradients
            accum_grads = [ag / GA_steps for ag in accum_grads]

            # Apply update
            optimizer.apply_gradients(zip(accum_grads, model.trainable_variables))

            # Reset accumulator
            accum_grads = [tf.zeros_like(var, dtype=tf.float32) for var in model.trainable_variables]

        # Track metric
        metric_per_sample = metric_fn(y_batch, y_pred)
        metric_value = tf.reduce_mean(metric_per_sample)
        train_metric_total += metric_value * len(x_batch)
        train_samples += len(x_batch)

    # Compute final average for the epoch
    train_metric_epoch = train_metric_total / train_samples
        
    # Validation
    for i in range(0, len(X_val), batch_size):
        x_val_batch = X_val[i:i+batch_size]
        y_val_batch = y_val[i:i+batch_size]
        
        val_preds = model(x_val_batch, training=False)
        
        val_metric_per_sample = metric_fn(y_val_batch, val_preds)
        val_metric_value = tf.reduce_mean(val_metric_per_sample)
        val_metric_total += val_metric_value * len(x_val_batch)
        val_samples += len(x_val_batch)
    
    val_metric_epoch = val_metric_total / val_samples
    
    print(f"Train metric: {train_metric_epoch:.4f} - Val metric: {val_metric_epoch:.4f}")
    
    # ReduceLROnPlateau
    if val_metric_epoch < best_val_loss:
        wait = 0
    else:
        wait += 1
        if wait >= reduce_lr_patience:
            old_lr = float(tf.keras.backend.get_value(optimizer.learning_rate))
            new_lr = old_lr * reduce_lr_factor
            optimizer.learning_rate.assign(new_lr)
            print(f">>> Reducing LR from {old_lr:.6f} to {new_lr:.6f}")
            wait = 0
    
    # Checkpoint
    if val_metric_epoch < best_val_loss:
        best_val_loss = val_metric_epoch
        model.save("Log_files/" + starting_datetime + "/Checkpoint_ResNet.keras")
        print(">>> Model checkpoint saved.")

    # Logging
    logs = {logger.monitor: val_metric_epoch}
    logger.on_epoch_end(epoch, logs)